# Diabetes Patients Diagnostic Classification

Diagnostic machine learning pipeline predicting diabetes onset based on clinical laboratory measurements.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [ ]:
df = pd.read_csv('diabetes_patients_classification_data.csv')
print('Shape:', df.shape)
display(df.head())

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x='outcome', data=df)
plt.title('Target Distribution: outcome')
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop(columns=['outcome'])
y = df['outcome']
num_cols = ['pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree_function', 'age']
cat_cols = []
preprocessor = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {'LogisticRegression': LogisticRegression(max_iter=1000), 'RandomForestClassifier': RandomForestClassifier(n_estimators=150, random_state=42), 'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42)}
best_score, best_pipe, best_name, results = -1, None, '', {}
for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    results[name] = {'accuracy': round(accuracy_score(y_test, preds), 4), 'f1_score': round(f1_score(y_test, preds), 4), 'roc_auc': round(roc_auc_score(y_test, probs), 4)}
    if results[name]['f1_score'] > best_score:
        best_score, best_pipe, best_name = results[name]['f1_score'], pipe, name
print('Best Model:', best_name)
print(pd.DataFrame(results).T)

In [ ]:
joblib.dump(best_pipe, 'best_model.pkl')
print('Saved best_model.pkl successfully.')